In [53]:
import os
import hashlib
import requests
from xml.etree import ElementTree as ET
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain.chat_models import init_chat_model

load_dotenv()

DB_DIR = "./chroma_db"
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL, LLM_PROVIDER = "llama3.2", "ollama"
ACT_URL = "https://www.legislation.gov.uk/ukpga/2006/26/data.xml"

if LLM_PROVIDER == "anthropic":
    assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY missing from .env"

In [54]:
NS = "{http://www.legislation.gov.uk/namespaces/legislation}"


def load(url):
    req = requests.get(url)
    if req.status_code != 200:
        return []

    root = ET.fromstring(req.content)
    docs = []

    for p1 in root.iter(f"{NS}P1"):
        section = "".join(p1.find(f"{NS}Pnumber").itertext()).strip()

        for p2 in p1.iter(f"{NS}P2"):
            uri = p2.get("DocumentURI")
            if uri is None:
                continue
            
            sub = "".join(p2.find(f"{NS}Pnumber").itertext()).strip()
            content = " ".join(
            t.strip() for el in p2.find(f"{NS}P2para").iter()
            if el.tag != f"{NS}Pnumber"
            for t in [el.text, el.tail] if t and t.strip()
            )

            docs.append(Document(
                page_content=f"Section {section}({sub}): {content}",
                metadata={"source": url, "section": section, "subsection": sub, "uri": uri},
            ))

    return docs

In [55]:
def chunk_id(doc):
    return hashlib.md5(doc.metadata["uri"].encode()).hexdigest()

def get_store():
    return Chroma(
        collection_name="legislation",
        embedding_function=OllamaEmbeddings(model=EMBED_MODEL),
        persist_directory=DB_DIR,
    )

In [56]:
def index(url=ACT_URL, rebuild=False):
    store = get_store()
    if rebuild:
        store.delete_collection()
        store = get_store()

    docs = load(url)
    store.add_documents(docs, ids=[chunk_id(d) for d in docs])
    print(f"{len(docs)} subsections indexed, store holds {store._collection.count()}")
    return store

In [57]:
def retrieve(store, question, k=4):
    return store.similarity_search_with_score(question, k=k)


def show(hits):
    for doc, score in hits:
        print(f"[{score:.3f}] s.{doc.metadata['section']}({doc.metadata['subsection']})")
        print("   ", doc.page_content[:200], "\n")

In [ ]:
llm = init_chat_model(LLM_MODEL, model_provider=LLM_PROVIDER)


def build_prompt(question, hits):
    ctx = "\n\n".join(d.page_content for d, _ in hits)
    return (f"Answer using only the context below. Cite the section you used. "
            f"If the context does not contain the answer, say so.\n\n"
            f"Context:\n{ctx}\n\nQuestion: {question}")


def ask(store, question, k=4, debug=False):
    hits = retrieve(store, question, k)
    if debug:
        show(hits)
    return llm.invoke(build_prompt(question, hits)).text

In [63]:
# Testing code
docs = load(ACT_URL)
assert docs, "loader returned nothing"
assert all(d.metadata["uri"] for d in docs), "missing URI"
assert len({d.metadata["uri"] for d in docs}) == len(docs), "duplicate URIs"
assert all(len(d.page_content.split(": ", 1)[1]) > 0 for d in docs), "empty content"
print(f"{len(docs)} docs, all valid")

382 docs, all valid


In [65]:
print(ask(store, "Can a right of common be severed from the land it is attached to?", debug=True))

print(ask(store, "What is the capital of France?"))

[0.270] s.9(1)
    Section 9(1): This section applies to a right of common which— is registered in a register of common land or town or village greens as attached to any land; and would, apart from this section, be capa 

[0.343] s.1(4)
    Section 1(4): Where a right of common to which section 9 applies is exercisable over land for which a commons council is established, the right may only be severed by a transfer under sub-paragraph (1 

[0.350] s.9(2)
    Section 9(2): A right of common to which this section applies is not at any time on or after the day on which this section comes into force capable of being severed from the land to which it is attach 

[0.363] s.6(3)
    Section 6(3): A right of common may be created over land to which this Part applies by way of express grant if— the land is not registered as a town or village green; and the right is attached to land 



ResponseError: model 'llama3.2' not found (status code: 404)